In [1]:
%%configure -f
{
  "defaultLakehouse": {
    "name": "lh_insurance_bronze_silver",
    "id": "c5d42bb8-2bbc-4172-a40b-8fbcdee30e29",
    "workspaceId": "3533e4eb-ba19-4350-b072-e6497106fb81"
  }
}

StatementMeta(, 0ffd0077-eb33-4def-9afa-96ec1761f776, -1, Finished, Available, Finished, True)

In [2]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

StatementMeta(, 0ffd0077-eb33-4def-9afa-96ec1761f776, 3, Finished, Available, Finished, False)

In [3]:
claims = spark.table("silver_stg_claims")
policies = spark.table("silver_stg_policies")
customers = spark.table("silver_stg_customers")
vehicles = spark.table("silver_stg_vehicles")
payments = spark.table("silver_stg_payments")

StatementMeta(, 0ffd0077-eb33-4def-9afa-96ec1761f776, 4, Finished, Available, Finished, False)

In [4]:
claims_hub_src = claims.withColumn(
    "claim_hk", F.sha2(F.upper(F.trim(F.col("claim_number"))), 256)
)

policies_hub_src = policies.withColumn(
    "policy_hk", F.sha2(F.upper(F.trim(F.col("policy_number"))), 256)
)

customers_hub_src = customers.withColumn(
    "customer_hk", F.sha2(F.upper(F.trim(F.col("customer_id"))), 256)
)

vehicles_hub_src = vehicles.withColumn(
    "vehicle_hk", F.sha2(F.upper(F.trim(F.col("vin"))), 256)
)

payments_hub_src = payments.withColumn(
    "payment_hk", F.sha2(F.upper(F.trim(F.col("payment_number"))), 256)
)

StatementMeta(, 0ffd0077-eb33-4def-9afa-96ec1761f776, 5, Finished, Available, Finished, False)

In [5]:
hub_claim = (
    claims_hub_src
    .select(
        "claim_hk",
        "claim_number",
        "ingestion_ts",
        "record_source"
    )
    .withColumnRenamed("ingestion_ts", "load_datetime")
    .dropDuplicates(["claim_hk"])
)

hub_policy = (
    policies_hub_src
    .select(
        "policy_hk",
        "policy_number",
        "ingestion_ts",
        "record_source"
    )
    .withColumnRenamed("ingestion_ts", "load_datetime")
    .dropDuplicates(["policy_hk"])
)

hub_customer = (
    customers_hub_src
    .select(
        "customer_hk",
        "customer_id",
        "ingestion_ts",
        "record_source"
    )
    .withColumnRenamed("ingestion_ts", "load_datetime")
    .dropDuplicates(["customer_hk"])
)

hub_vehicle = (
    vehicles_hub_src
    .select(
        "vehicle_hk",
        "vin",
        "ingestion_ts",
        "record_source"
    )
    .withColumnRenamed("ingestion_ts", "load_datetime")
    .dropDuplicates(["vehicle_hk"])
)

hub_payment = (
    payments_hub_src
    .select(
        "payment_hk",
        "payment_number",
        "ingestion_ts",
        "record_source"
    )
    .withColumnRenamed("ingestion_ts", "load_datetime")
    .dropDuplicates(["payment_hk"])
)

StatementMeta(, 0ffd0077-eb33-4def-9afa-96ec1761f776, 6, Finished, Available, Finished, False)

In [6]:
link_claim_policy = (
    claims_hub_src
    .withColumn("policy_hk", F.sha2(F.upper(F.trim(F.col("policy_number"))), 256))
    .withColumn("claim_policy_hk", F.sha2(F.concat_ws("|", F.col("claim_hk"), F.col("policy_hk")), 256))
    .select(
        "claim_policy_hk",
        "claim_hk",
        "policy_hk",
        F.col("ingestion_ts").alias("load_datetime"),
        "record_source"
    )
    .dropDuplicates(["claim_policy_hk"])
)

link_policy_customer = (
    policies_hub_src
    .withColumn("customer_hk", F.sha2(F.upper(F.trim(F.col("customer_id"))), 256))
    .withColumn("policy_customer_hk", F.sha2(F.concat_ws("|", F.col("policy_hk"), F.col("customer_hk")), 256))
    .select(
        "policy_customer_hk",
        "policy_hk",
        "customer_hk",
        F.col("ingestion_ts").alias("load_datetime"),
        "record_source"
    )
    .dropDuplicates(["policy_customer_hk"])
)

link_claim_vehicle = (
    claims_hub_src
    .withColumn("vehicle_hk", F.sha2(F.upper(F.trim(F.col("vin"))), 256))
    .withColumn("claim_vehicle_hk", F.sha2(F.concat_ws("|", F.col("claim_hk"), F.col("vehicle_hk")), 256))
    .select(
        "claim_vehicle_hk",
        "claim_hk",
        "vehicle_hk",
        F.col("ingestion_ts").alias("load_datetime"),
        "record_source"
    )
    .dropDuplicates(["claim_vehicle_hk"])
)

link_claim_payment = (
    payments_hub_src
    .withColumn("claim_hk", F.sha2(F.upper(F.trim(F.col("claim_number"))), 256))
    .withColumn("claim_payment_hk", F.sha2(F.concat_ws("|", F.col("claim_hk"), F.col("payment_hk")), 256))
    .select(
        "claim_payment_hk",
        "claim_hk",
        "payment_hk",
        F.col("ingestion_ts").alias("load_datetime"),
        "record_source"
    )
    .dropDuplicates(["claim_payment_hk"])
)

StatementMeta(, 0ffd0077-eb33-4def-9afa-96ec1761f776, 7, Finished, Available, Finished, False)

In [7]:
sat_claim_details = (
    claims_hub_src
    .select(
        "claim_hk",
        F.col("ingestion_ts").alias("load_datetime"),
        F.col("reported_date").cast("timestamp").alias("effective_from"),
        "incident_date",
        "reported_date",
        "claim_status",
        "loss_type",
        "claim_amount",
        "reserve_amount",
        "fraud_indicator",
        F.col("claim_hashdiff").alias("hashdiff"),
        "record_source"
    )
    .dropDuplicates(["claim_hk", "hashdiff"])
)

sat_policy_coverage = (
    policies_hub_src
    .select(
        "policy_hk",
        F.col("ingestion_ts").alias("load_datetime"),
        F.col("policy_start_date").cast("timestamp").alias("effective_from"),
        "policy_start_date",
        "policy_end_date",
        "coverage_type",
        "deductible_amount",
        "underwriting_status",
        "state_code",
        F.col("policy_hashdiff").alias("hashdiff"),
        "record_source"
    )
    .dropDuplicates(["policy_hk", "hashdiff"])
)

sat_customer_profile = (
    customers_hub_src
    .select(
        "customer_hk",
        F.col("ingestion_ts").alias("load_datetime"),
        F.current_timestamp().alias("effective_from"),
        "full_name",
        "date_of_birth",
        "phone_number",
        "email",
        "address_line_1",
        "city",
        "state_code",
        "postal_code",
        "risk_segment",
        F.col("customer_hashdiff").alias("hashdiff"),
        "record_source"
    )
    .dropDuplicates(["customer_hk", "hashdiff"])
)

sat_vehicle_attributes = (
    vehicles_hub_src
    .select(
        "vehicle_hk",
        F.col("ingestion_ts").alias("load_datetime"),
        F.current_timestamp().alias("effective_from"),
        "vehicle_make",
        "vehicle_model",
        "vehicle_year",
        "usage_type",
        "garaging_state",
        F.col("vehicle_hashdiff").alias("hashdiff"),
        "record_source"
    )
    .dropDuplicates(["vehicle_hk", "hashdiff"])
)

sat_payment_details = (
    payments_hub_src
    .select(
        "payment_hk",
        F.col("ingestion_ts").alias("load_datetime"),
        F.col("payment_date").cast("timestamp").alias("effective_from"),
        "payment_date",
        "payment_amount",
        "payment_method",
        "payment_status",
        F.col("payment_hashdiff").alias("hashdiff"),
        "record_source"
    )
    .dropDuplicates(["payment_hk", "hashdiff"])
)

StatementMeta(, 0ffd0077-eb33-4def-9afa-96ec1761f776, 8, Finished, Available, Finished, False)

In [8]:
hub_claim.write.mode("overwrite").format("delta").saveAsTable("hub_claim")
hub_policy.write.mode("overwrite").format("delta").saveAsTable("hub_policy")
hub_customer.write.mode("overwrite").format("delta").saveAsTable("hub_customer")
hub_vehicle.write.mode("overwrite").format("delta").saveAsTable("hub_vehicle")
hub_payment.write.mode("overwrite").format("delta").saveAsTable("hub_payment")

link_claim_policy.write.mode("overwrite").format("delta").saveAsTable("link_claim_policy")
link_policy_customer.write.mode("overwrite").format("delta").saveAsTable("link_policy_customer")
link_claim_vehicle.write.mode("overwrite").format("delta").saveAsTable("link_claim_vehicle")
link_claim_payment.write.mode("overwrite").format("delta").saveAsTable("link_claim_payment")

sat_claim_details.write.mode("overwrite").format("delta").saveAsTable("sat_claim_details")
sat_policy_coverage.write.mode("overwrite").format("delta").saveAsTable("sat_policy_coverage")
sat_customer_profile.write.mode("overwrite").format("delta").saveAsTable("sat_customer_profile")
sat_vehicle_attributes.write.mode("overwrite").format("delta").saveAsTable("sat_vehicle_attributes")
sat_payment_details.write.mode("overwrite").format("delta").saveAsTable("sat_payment_details")

StatementMeta(, 0ffd0077-eb33-4def-9afa-96ec1761f776, 9, Finished, Available, Finished, False)

In [9]:
for table_name in [
    "hub_claim", "hub_policy", "hub_customer", "hub_vehicle", "hub_payment",
    "link_claim_policy", "link_policy_customer", "link_claim_vehicle", "link_claim_payment",
    "sat_claim_details", "sat_policy_coverage", "sat_customer_profile", "sat_vehicle_attributes", "sat_payment_details"
]:
    print(f"\n=== {table_name} ===")
    spark.sql(f"SELECT COUNT(*) AS row_count FROM {table_name}").show()

StatementMeta(, 0ffd0077-eb33-4def-9afa-96ec1761f776, 10, Finished, Available, Finished, False)


=== hub_claim ===
+---------+
|row_count|
+---------+
|       12|
+---------+


=== hub_policy ===
+---------+
|row_count|
+---------+
|        8|
+---------+


=== hub_customer ===
+---------+
|row_count|
+---------+
|        8|
+---------+


=== hub_vehicle ===
+---------+
|row_count|
+---------+
|        8|
+---------+


=== hub_payment ===
+---------+
|row_count|
+---------+
|       10|
+---------+


=== link_claim_policy ===
+---------+
|row_count|
+---------+
|       12|
+---------+


=== link_policy_customer ===
+---------+
|row_count|
+---------+
|        8|
+---------+


=== link_claim_vehicle ===
+---------+
|row_count|
+---------+
|       12|
+---------+


=== link_claim_payment ===
+---------+
|row_count|
+---------+
|       10|
+---------+


=== sat_claim_details ===
+---------+
|row_count|
+---------+
|       12|
+---------+


=== sat_policy_coverage ===
+---------+
|row_count|
+---------+
|        8|
+---------+


=== sat_customer_profile ===
+---------+
|row_count|
+--